# H2MM on simulated smFRET data

This tutorial simulates a two-state single-molecule FRET experiment, carries the
photons through a real `tttrlib.TTTR` object, and fits a **photon-by-photon
Hidden Markov Model (H2MM)** to recover the FRET states, their kinetics, and the
per-photon state trajectory.

We use synthetic data so the notebook is self-contained and the ground truth is
known — the same steps apply to real `.bur` burst folders.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tttrlib

from chisurf.plugins.burst.burst_h2mm.core import h2mm, analysis, export
from chisurf.plugins.burst.burst_h2mm.core.photons import StreamDef, bursts_from_dataframe
import pandas as pd

## 1. Define a ground-truth two-state model

State 0 is a low-FRET state (mostly donor / stream 0), state 1 is high-FRET
(mostly acceptor / stream 1). The transition matrix is for **one macro-time
tick**; off-diagonal terms are the per-tick switching probabilities.

In [ ]:
ground_truth = h2mm.H2mmModel(
    prior=np.array([0.5, 0.5]),
    trans=np.array([[0.995, 0.005],
                    [0.010, 0.990]]),
    obs=np.array([[0.85, 0.15],    # state 0: E = 0.15
                  [0.20, 0.80]]),  # state 1: E = 0.80
)
E_true = ground_truth.obs[:, 1] / ground_truth.obs.sum(1)
print("true per-state FRET E:", np.round(E_true, 3))

## 2. Simulate bursts and pack them into a `tttrlib.TTTR`

Each burst is a train of photons with Poisson-distributed inter-photon times
(variable Δt — the situation H2MM is built for). We simulate the hidden state
trajectory and emitted colours, then append everything to a real TTTR object so
the notebook exercises the exact same extraction path used on measured data.

In [ ]:
rng = np.random.default_rng(1)
n_bursts, burst_len = 400, 120

times = [np.concatenate([[0], np.cumsum(rng.poisson(4, burst_len - 1) + 1)]).astype(np.int64)
         for _ in range(n_bursts)]
streams = h2mm.simulate_bursts(ground_truth, times, seed=2)

macro, chan, rows = [], [], []
offset, base = 0, 0
for t, s in zip(times, streams):
    macro.append((t + base).astype(np.uint64))
    chan.append(s.astype(np.int8))                 # channel 0 = donor, 1 = acceptor
    rows.append(("sim.spc", offset, offset + len(t)))
    offset += len(t)
    base += int(t[-1]) + 100000                    # large gap keeps bursts distinct

macro = np.concatenate(macro).astype(np.uint64)
chan = np.concatenate(chan).astype(np.int8)
micro = np.zeros(macro.size, dtype=np.uint16)
tttr = tttrlib.TTTR()
tttr.append_events(macro, micro, chan, np.zeros(macro.size, np.int8), False, 0)
print(f"{n_bursts} bursts, {macro.size} photons")

## 3. Extract engine-ready burst photons

`.bur` burst tables carry `First File` / `First Photon` / `Last Photon` indices
into the TTTR. We build that table, define the donor/acceptor streams, and
extract — asking for the per-photon metadata so we can build result tables
later.

In [ ]:
df = pd.DataFrame(rows, columns=["First File", "First Photon", "Last Photon"])
stream_defs = [StreamDef("donor", [0]), StreamDef("acceptor", [1])]

data, meta = bursts_from_dataframe(df, {"sim.spc": tttr}, stream_defs,
                                   min_photons=5, return_meta=True)
print(f"engine layout: {data.n_bursts} bursts, {data.n_photons} photons, "
      f"{data.unique_dt.shape[0]} unique dt")

## 4. Fit and select the number of states

`analyze` fits every state count in the range, scores BIC/ICL, and selects the
best. On a clean two-state dataset BIC bottoms out at 2. We plot the
model-selection curve.

In [ ]:
ana = analysis.analyze(data, state_counts=(1, 2, 3, 4), criterion="bic",
                       base_time_s=1e-6, n_restarts=2, max_iter=500)
print("selected states:", ana.best.n_states)

ks = [f.n_states for f in ana.scan]
bics = [f.bic for f in ana.scan]
plt.figure(figsize=(4, 3))
plt.plot(ks, bics, "o-")
plt.axvline(ana.best.n_states, color="r", ls="--", label=f"best = {ana.best.n_states}")
plt.xlabel("number of states"); plt.ylabel("BIC"); plt.legend(); plt.tight_layout()
plt.show()

## 5. Recovered FRET states and kinetics

Compare the fitted per-state FRET and transition rates to the ground truth.

In [ ]:
E_fit = np.sort(ana.fret)
print("true  E:", np.round(np.sort(E_true), 3))
print("fit   E:", np.round(E_fit, 3))

plt.figure(figsize=(4, 3))
plt.bar(range(len(ana.fret)), np.sort(ana.fret), width=0.5)
plt.xticks(range(len(ana.fret)), [f"state {i}" for i in range(len(ana.fret))])
plt.ylabel("apparent FRET E"); plt.ylim(0, 1); plt.tight_layout()
plt.show()

## 6. Per-photon Viterbi state trajectory

The Viterbi path assigns each photon its most-likely hidden state. We show the
trajectory for one burst — the sub-burst dynamics H2MM resolves.

In [ ]:
path, icl = h2mm.viterbi(ana.best.model, data)
b = 0
s, e = int(data.burst_offsets[b]), int(data.burst_offsets[b + 1])
t_rel = (meta.macro_time[s:e] - meta.macro_time[s]) * 1e-6 * 1e3  # ms

plt.figure(figsize=(6, 2.2))
plt.step(t_rel, path[s:e], where="post")
plt.scatter(t_rel, data.streams[s:e] * 0.0 - 0.3, c=data.streams[s:e],
            cmap="coolwarm", s=8, label="photon colour")
plt.yticks([0, 1], ["state 0", "state 1"]); plt.xlabel("time in burst (ms)")
plt.title("Viterbi state path (one burst)"); plt.tight_layout()
plt.show()

## 7. Compare the compute engines

The exact `em` engine is the default. `em-float32` and the amortised neural
`surrogate` trade a little accuracy for speed. Here we time the exact and
float32 engines (the surrogate needs a pre-trained model — see the docs).

In [ ]:
import time
for engine in ["em", "em-float32"]:
    t0 = time.perf_counter()
    a = analysis.analyze(data, state_counts=(2,), engine=engine, n_restarts=1, max_iter=500)
    dt = time.perf_counter() - t0
    print(f"{engine:12s}: {dt*1000:6.1f} ms   E = {np.round(np.sort(a.fret), 3)}")

## 8. Export tables for ndxplorer (ndX)

The per-photon and per-burst tables are plain numeric tables that open directly
in ndX. Colour the per-photon scatter by `State` to view the recovered
trajectory across the whole dataset.

In [ ]:
import tempfile, pathlib
tables = export.build_tables(data, meta, path, ana.fret, base_time_s=1e-6)
out = pathlib.Path(tempfile.mkdtemp())
tables.photons.to_csv(out / "h2mm_photons.csv", index=False)   # open in ndX (CSV reader)
tables.bursts.to_csv(out / "h2mm_bursts.csv", index=False)
print("per-photon columns:", list(tables.photons.columns))
tables.photons.head()

## 9. Appendix — generate an on-disk example dataset

Everything above ran in memory. The plugin also ships a generator that writes a
**real, loadable dataset** — a Photon-HDF5 `tttrlib` file plus a `.bur` burst
table — so you can exercise the GUI and the `h2mm` CLI on genuine files.

In [ ]:
from chisurf.plugins.burst.burst_h2mm.examples.generate_example_data import (
    generate_example_data,
)
ex_dir = pathlib.Path(tempfile.mkdtemp()) / "h2mm_example"
bur, tttr = generate_example_data(ex_dir, n_bursts=200, burst_len=100, seed=1)
print("wrote:", bur.name, "and", tttr.name)
print("analyse from the command line:")
print(f"    h2mm compute {ex_dir} --file-type auto")